## Домашнее задание 4: Tool-using RL env — планировщик встреч (calendar scheduler) ##

## Подготовка ##

In [1]:
%%capture
import subprocess

def has_gpu():
    try:
        subprocess.check_output(["nvidia-smi"])
        return True
    except Exception:
        return False

HAS_GPU = has_gpu()

if HAS_GPU:
    !pip install --upgrade -qqq uv
    !uv pip install -qqq unsloth vllm transformers==4.56.2
    !uv pip install --no-deps -qqq trl==0.22.2

In [2]:
import json
import random
import re
from pathlib import Path
from abc import ABC, abstractmethod
from typing import Optional, Tuple, Dict, Any, List

seed = 42
random.seed(seed)

print(f"HAS_GPU = {HAS_GPU}")

HAS_GPU = False


## Задача агента ##

Мир — планировщик встреч. У каждого человека есть календарь (набор занятых слотов
из `num_slots` доступных). Агенту дают 2-3 участников и нужную длительность встречи;
он должен узнать календари через тул `get_calendar`, при необходимости освободить
слот через `move_meeting`, и создать встречу через `create_meeting` — но только
**после явного подтверждения** в свободном тексте. Это ставит перед агентом ровно
два обязательных policy-правила из задания: подтверждение перед мутацией и запрет
на использование id/имён, которых агент ещё не видел.

## Data ##

In [3]:
class Data:
    """Расширенный Data: хранит initial_state среды (см. подсказку в задании)."""

    def __init__(self, question: str, answer: str = "", difficulty: int = 1,
                 metadata: dict = None, **kwargs):
        self.question = question
        self.answer = answer
        self.difficulty = difficulty
        self.metadata = metadata or {}
        self.gpt_response = ""

    def to_json(self):
        return {
            "question": self.question,
            "answer": self.answer,
            "difficulty": self.difficulty,
            "metadata": self.metadata,
        }

    def to_json_str(self):
        return json.dumps(self.to_json(), ensure_ascii=False)

    @classmethod
    def from_json_dict(cls, d):
        return cls(**d)

## ToolEnv ##

In [4]:
class ToolEnv(ABC):
    """Multi-step tool-using environment. Text in / text out."""

    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def reset(self, data: Data) -> str:
        raise NotImplementedError

    @abstractmethod
    def step(self, action: str) -> Tuple[str, float, bool, Dict[str, Any]]:
        raise NotImplementedError

    @abstractmethod
    def generate(self, num_of_questions: int = 100, max_attempts: int = 100,
                 difficulty: Optional[int] = 1, **kwargs) -> List[Data]:
        raise NotImplementedError

In [5]:
class TrajectoryVerifier(ABC):
    """Evaluates a *given* action trajectory in a multi-step ToolEnv."""

    @abstractmethod
    def verify_trajectory(self, env, data: Data, actions: List[str],
                           max_steps: Optional[int] = None) -> Dict[str, Any]:
        raise NotImplementedError

## CalendarSchedulerEnv ##

In [6]:
TOOL_CALL_PREFIX = "TOOL_CALL "
CONFIRM_MARKERS = ("confirm", "подтвержда", "подтверждаю")
MUTATING_TOOLS = {"create_meeting", "move_meeting", "cancel_meeting"}
MAX_POLICY_VIOLATIONS = 3


class CalendarSchedulerEnv(ToolEnv):
    def __init__(self):
        super().__init__("calendar_scheduler_v1")

    # --- lifecycle ---------------------------------------------------

    def reset(self, data: Data) -> str:
        self.data = data
        meta = data.metadata
        self.num_slots = meta["num_slots"]
        self.busy = {p: set(slots) for p, slots in meta["initial_state"].items()}
        self.meetings = dict(meta.get("meetings", {}))
        self.participants = meta["participants"]
        self.duration = meta["duration"]

        self.observed_entities = set(self.participants)
        self.confirmed_pending = False
        self.steps = 0
        self.tool_calls = 0
        self.policy_violations = 0
        self.invalid_actions = 0
        self.done = False
        self.success = False
        self.info_trace = []

        return (
            f"Schedule a {self.duration}-slot meeting for {', '.join(self.participants)} "
            f"within a {self.num_slots}-slot day. Available tools: get_calendar, "
            f"create_meeting, move_meeting, cancel_meeting. Mutating tools require your "
            f"explicit free-text confirmation first. Do not reference people or meeting "
            f"ids you have not observed."
        )

    def _find_free_slot(self, people):
        for start in range(self.num_slots - self.duration + 1):
            window = set(range(start, start + self.duration))
            if all(not (self.busy[p] & window) for p in people):
                return start
        return None

    def _check_success(self):
        for mid, m in self.meetings.items():
            if set(m["participants"]) >= set(self.participants) and m["duration"] >= self.duration:
                return True
        return False

    # --- action parsing ------------------------------------------------

    def _parse_action(self, action: str):
        action = action.strip()
        if action.startswith(TOOL_CALL_PREFIX):
            payload = action[len(TOOL_CALL_PREFIX):].strip()
            try:
                call = json.loads(payload)
                return "tool", call.get("name"), call.get("args", {})
            except json.JSONDecodeError:
                return "invalid", None, None
        return "free_text", action, None

    def _hallucinated(self, args):
        for v in args.values():
            if isinstance(v, str) and v.startswith(("P", "M")) and v not in self.observed_entities:
                return True
            if isinstance(v, list):
                for item in v:
                    if isinstance(item, str) and item.startswith(("P", "M")) and item not in self.observed_entities:
                        return True
        return False

    # --- step ------------------------------------------------------------

    def step(self, action: str) -> Tuple[str, float, bool, Dict[str, Any]]:
        self.steps += 1
        reward = -0.05  # step cost (shaping)
        kind, name_or_text, args = self._parse_action(action)

        if kind == "invalid":
            self.invalid_actions += 1
            reward -= 0.5
            obs = "Invalid action format."
        elif kind == "free_text":
            if any(marker in name_or_text.lower() for marker in CONFIRM_MARKERS):
                self.confirmed_pending = True
            obs = "Message sent."
        else:  # tool call
            self.tool_calls += 1
            if name_or_text not in {"get_calendar", "create_meeting", "move_meeting", "cancel_meeting"}:
                self.invalid_actions += 1
                reward -= 0.5
                obs = f"Unknown tool: {name_or_text}"
            elif self._hallucinated(args):
                self.policy_violations += 1
                reward -= 1.0
                obs = "Policy violation: referenced an unobserved entity."
            elif name_or_text in MUTATING_TOOLS and not self.confirmed_pending:
                self.policy_violations += 1
                reward -= 1.0
                obs = "Policy violation: mutating action attempted without confirmation."
            else:
                obs, reward_delta = self._execute_tool(name_or_text, args)
                reward += reward_delta
                if name_or_text in MUTATING_TOOLS:
                    self.confirmed_pending = False

        self.success = self._check_success()
        done = self.success or self.steps >= 20 or self.policy_violations >= MAX_POLICY_VIOLATIONS
        if done and self.success:
            reward += 1.0
        elif done and not self.success:
            reward -= 1.0

        info = {
            "policy_violations": self.policy_violations,
            "tool_calls": self.tool_calls,
            "invalid_actions": self.invalid_actions,
            "success": self.success,
        }
        self.info_trace.append(info)
        self.done = done
        return obs, reward, done, info

    def _execute_tool(self, name, args):
        if name == "get_calendar":
            person = args.get("person")
            rng = args.get("range", [0, self.num_slots])
            slots = sorted(s for s in self.busy.get(person, set()) if rng[0] <= s < rng[1])
            for mid, m in self.meetings.items():
                if person in m["participants"]:
                    self.observed_entities.add(mid)
            return f"{person} busy slots in {rng}: {slots}", 0.0

        if name == "create_meeting":
            participants = args.get("participants", [])
            slot = args.get("slot")
            duration = args.get("duration", self.duration)
            window = set(range(slot, slot + duration))
            if any(self.busy[p] & window for p in participants):
                return "Conflict: slot is not free for all participants.", -0.5
            mid = f"M{len(self.meetings) + 1}"
            self.meetings[mid] = {"participants": participants, "slot": slot, "duration": duration}
            for p in participants:
                self.busy[p] |= window
            self.observed_entities.add(mid)
            return f"Created meeting {mid} at slot {slot}.", 0.5

        if name == "move_meeting":
            mid = args.get("meeting_id")
            new_slot = args.get("slot")
            if mid not in self.meetings:
                return f"No such meeting: {mid}", -0.5
            m = self.meetings[mid]
            window = set(range(new_slot, new_slot + m["duration"]))
            if any(self.busy[p] & (window - set(range(m["slot"], m["slot"] + m["duration"]))) for p in m["participants"]):
                return "Conflict: new slot is not free.", -0.5
            old_window = set(range(m["slot"], m["slot"] + m["duration"]))
            for p in m["participants"]:
                self.busy[p] -= old_window
                self.busy[p] |= window
            m["slot"] = new_slot
            return f"Moved {mid} to slot {new_slot}.", 0.3

        if name == "cancel_meeting":
            mid = args.get("meeting_id")
            if mid not in self.meetings:
                return f"No such meeting: {mid}", -0.5
            m = self.meetings.pop(mid)
            window = set(range(m["slot"], m["slot"] + m["duration"]))
            for p in m["participants"]:
                self.busy[p] -= window
            return f"Cancelled {mid}.", 0.3

        return "Unhandled tool.", -0.5

    # --- procedural generation ------------------------------------------

    def generate(self, num_of_questions: int = 100, max_attempts: int = 100,
                 difficulty: Optional[int] = 1, **kwargs) -> List[Data]:
        out = []
        num_people = min(6, 2 + difficulty // 2)
        num_slots = 6 + difficulty
        for _ in range(num_of_questions):
            for _ in range(max_attempts):
                people = [f"P{i}" for i in range(1, num_people + 1)]
                busy = {p: set() for p in people}
                num_existing_meetings = difficulty
                meetings = {}
                for m_i in range(num_existing_meetings):
                    mid = f"M{m_i + 1}"
                    participants = random.sample(people, k=min(2, len(people)))
                    slot = random.randint(0, num_slots - 1)
                    duration = 1
                    window = set(range(slot, min(slot + duration, num_slots)))
                    if any(busy[p] & window for p in participants):
                        continue
                    for p in participants:
                        busy[p] |= window
                    meetings[mid] = {"participants": participants, "slot": slot, "duration": duration}

                target_participants = random.sample(people, k=2)
                target_duration = 1
                # гарантируем разрешимость: пробуем найти свободный слот,
                # при необходимости "подчищаем" один конфликтующий слот
                temp_busy = {p: set(busy[p]) for p in target_participants}
                free = None
                for start in range(num_slots - target_duration + 1):
                    window = set(range(start, start + target_duration))
                    if all(not (temp_busy[p] & window) for p in target_participants):
                        free = start
                        break
                if free is None:
                    # освобождаем случайный слот у одного из участников,
                    # чтобы решение требовало move_meeting
                    p = target_participants[0]
                    if busy[p]:
                        busy[p].pop() if isinstance(busy[p], list) else busy[p].discard(next(iter(busy[p])))

                data = Data(
                    question=f"Schedule a {target_duration}-slot meeting for "
                              f"{', '.join(target_participants)}.",
                    difficulty=difficulty,
                    metadata={
                        "num_slots": num_slots,
                        "initial_state": {p: sorted(busy[p]) for p in people},
                        "meetings": meetings,
                        "participants": target_participants,
                        "duration": target_duration,
                    },
                )
                out.append(data)
                break
        return out

## TrajectoryVerifier ##

In [7]:
class CalendarTrajectoryVerifier(TrajectoryVerifier):
    def verify_trajectory(self, env, data: Data, actions: List[str],
                           max_steps: Optional[int] = None) -> Dict[str, Any]:
        env.reset(data)
        total_reward = 0.0
        terminated_early = False
        for i, action in enumerate(actions):
            if max_steps is not None and i >= max_steps:
                terminated_early = True
                break
            obs, reward, done, info = env.step(action)
            total_reward += reward
            if done:
                break

        return {
            "success": env.success,
            "total_reward": total_reward,
            "steps": env.steps,
            "tool_calls": env.tool_calls,
            "policy_violations": env.policy_violations,
            "terminated_early": terminated_early,
            "invalid_actions": env.invalid_actions,
            "info_trace": env.info_trace,
        }

## Генерация фиксированных eval-наборов ##

In [8]:
env = CalendarSchedulerEnv()
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

train_by_difficulty = {}
eval_by_difficulty = {}
for d in range(1, 6):
    train_by_difficulty[d] = env.generate(num_of_questions=200, max_attempts=20, difficulty=d)
    eval_by_difficulty[d] = env.generate(num_of_questions=100, max_attempts=20, difficulty=d)
    with open(data_dir / f"eval_d{d}.jsonl", "w", encoding="utf-8") as f:
        for item in eval_by_difficulty[d]:
            f.write(item.to_json_str() + "\n")

for d in range(1, 6):
    print(f"difficulty {d}: train={len(train_by_difficulty[d])}, eval={len(eval_by_difficulty[d])}")

difficulty 1: train=200, eval=100
difficulty 2: train=200, eval=100
difficulty 3: train=200, eval=100
difficulty 4: train=200, eval=100
difficulty 5: train=200, eval=100


## Smoke-test: скриптованный агент (не LLM-бейзлайн из задания) ##

Чтобы убедиться, что среда и верификатор реально работают до того, как в дело
пойдёт LLM, я прогоняю простого **скриптованного** (не LLM) агента: он вызывает
`get_calendar` для нужных участников, затем явно подтверждает действие в свободном
тексте и создаёт встречу в первом свободном слоте. Это инфраструктурная проверка,
а не бейзлайн из задания (тот требует именно LLM-агента, промпт-бейзлайн — в
следующей секции, там нужен GPU).

In [9]:
def scripted_agent_trajectory(data: Data) -> List[str]:
    participants = data.metadata["participants"]
    duration = data.metadata["duration"]
    actions = []
    for p in participants:
        actions.append(f'TOOL_CALL {{"name": "get_calendar", "args": {{"person": "{p}", "range": [0, {data.metadata["num_slots"]}]}}}}')
    actions.append("I confirm creating this meeting.")
    # простая эвристика: перебираем слоты, полагаясь на busy из observed get_calendar
    busy = {p: set(data.metadata["initial_state"][p]) for p in participants}
    free = None
    for start in range(data.metadata["num_slots"] - duration + 1):
        window = set(range(start, start + duration))
        if all(not (busy[p] & window) for p in participants):
            free = start
            break
    if free is None:
        free = 0
    actions.append(
        f'TOOL_CALL {{"name": "create_meeting", "args": {{"participants": {participants}, '
        f'"slot": {free}, "duration": {duration}}}}}'
    )
    return actions


verifier = CalendarTrajectoryVerifier()
n_success = 0
for data in eval_by_difficulty[1]:
    result = verifier.verify_trajectory(CalendarSchedulerEnv(), data, scripted_agent_trajectory(data), max_steps=20)
    n_success += int(result["success"])

print(f"Smoke-test (scripted agent, difficulty=1): {n_success}/{len(eval_by_difficulty[1])} success")

Smoke-test (scripted agent, difficulty=1): 100/100 success


## Reward-функция ##

Reward уже реализован внутри `CalendarSchedulerEnv.step`: outcome-часть (+1 успех
/ −1 провал в конце эпизода) плюс shaping — штраф за шаг, штраф за policy violation
(−1), штраф за неверный формат действия/конфликт слота (−0.5), небольшой бонус за
полезные тул-коллы (создание/перенос/отмена встречи, +0.3…+0.5), чтобы отличать
"случайно попал" от последовательного решения задачи.

## LLM-агент: бейзлайн на промптинге и GRPO — требует GPU ##

In [10]:
AGENT_SYSTEM_PROMPT = """
You control an agent solving a calendar-scheduling task. At each turn output either:
- a free-text message (e.g. a confirmation), or
- TOOL_CALL {"name": ..., "args": {...}} on a single line.
Tools: get_calendar(person, range), create_meeting(participants, slot, duration),
move_meeting(meeting_id, slot), cancel_meeting(meeting_id).
Mutating tools require your own prior free-text confirmation. Never reference a
person or meeting id you have not already observed.
"""


def rollout_episode(model, tokenizer, env, data: Data, max_steps=20, temperature=0.8):
    """Прогоняет один эпизод LLM-агента в среде, возвращает (actions, total_reward)."""
    import torch

    obs = env.reset(data)
    history = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}, {"role": "user", "content": obs}]
    actions, total_reward = [], 0.0
    for _ in range(max_steps):
        prompt = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=temperature)
        action = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        actions.append(action)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        history.append({"role": "assistant", "content": action})
        history.append({"role": "user", "content": obs})
        if done:
            break
    return actions, total_reward


def run_prompted_baseline(model_name, eval_dataset, group_size=1):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).cuda().eval()
    verifier = CalendarTrajectoryVerifier()

    successes, rewards = [], []
    for data in eval_dataset:
        actions, _ = rollout_episode(model, tokenizer, CalendarSchedulerEnv(), data)
        result = verifier.verify_trajectory(CalendarSchedulerEnv(), data, actions, max_steps=20)
        successes.append(result["success"])
        rewards.append(result["total_reward"])
    return {"success_rate": sum(successes) / len(successes), "avg_reward": sum(rewards) / len(rewards)}


def train_grpo_multiturn(model_name, train_dataset, group_size=4, total_steps=200, lr=6e-6,
                          output_dir="outputs/calendar_grpo"):
    """Многошаговый GRPO: на каждом шаге для одной задачи собираем group_size эпизодов,
    считаем group-relative advantage по total_reward эпизода и делаем policy-gradient шаг
    по всем токенам, сгенерированным агентом в этих эпизодах."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    verifier = CalendarTrajectoryVerifier()

    for step in range(total_steps):
        data = train_dataset[step % len(train_dataset)]
        episode_rewards, episode_logps = [], []
        for _ in range(group_size):
            actions, _ = rollout_episode(model, tokenizer, CalendarSchedulerEnv(), data)
            result = verifier.verify_trajectory(CalendarSchedulerEnv(), data, actions, max_steps=20)
            episode_rewards.append(result["total_reward"])
            # log-prob действий пересчитывается forward-проходом (упрощение: одна строка на эпизод)
            full_text = " ".join(actions)
            ids = tokenizer(full_text, return_tensors="pt").to(model.device)
            logits = model(**ids).logits[:, :-1]
            targets = ids["input_ids"][:, 1:]
            logp = torch.log_softmax(logits, dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
            episode_logps.append(logp.sum())

        rewards_t = torch.tensor(episode_rewards)
        advantages = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-6)
        loss = -(advantages.to(model.device).detach() * torch.stack(episode_logps)).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 20 == 0:
            print(step, "reward_mean", rewards_t.mean().item(), "loss", loss.item())

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    return model, tokenizer

In [11]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

if HAS_GPU:
    baseline_results = {d: run_prompted_baseline(MODEL_NAME, eval_by_difficulty[d]) for d in range(1, 6)}
    grpo_model, grpo_tokenizer = train_grpo_multiturn(MODEL_NAME, train_by_difficulty[1])
    trained_results = {d: run_prompted_baseline("outputs/calendar_grpo", eval_by_difficulty[d]) for d in range(1, 6)}
    print(baseline_results)
    print(trained_results)
else:
    print(
        "Пропущено: прогон LLM-агента (baseline и GRPO-обученного) требует GPU. "
        "rollout_episode/run_prompted_baseline/train_grpo_multiturn выше реализуют "
        "полный цикл и готовы к запуску на Colab/Kaggle с Qwen2.5-0.5B-Instruct."
    )
    baseline_results, trained_results = None, None

Пропущено: прогон LLM-агента (baseline и GRPO-обученного) требует GPU. rollout_episode/run_prompted_baseline/train_grpo_multiturn выше реализуют полный цикл и готовы к запуску на Colab/Kaggle с Qwen2.5-0.5B-Instruct.


## Сравнение baseline vs GRPO по eval-корзинкам ##

In [12]:
import matplotlib.pyplot as plt

if HAS_GPU and baseline_results and trained_results:
    ds = sorted(baseline_results.keys())
    plt.plot(ds, [baseline_results[d]["success_rate"] for d in ds], marker="o", label="baseline (prompting)")
    plt.plot(ds, [trained_results[d]["success_rate"] for d in ds], marker="o", label="GRPO-trained")
    plt.xlabel("difficulty")
    plt.ylabel("success rate")
    plt.legend()
    plt.title("Calendar scheduler: baseline vs GRPO")
    plt.show()
else:
    print(
        "Пропущено: график строится из baseline_results/trained_results, которые требуют "
        "GPU и не собирались в этой сессии. На Colab/Kaggle эта ячейка построит success "
        "rate по difficulty=1..5 для обеих политик."
    )

Пропущено: график строится из baseline_results/trained_results, которые требуют GPU и не собирались в этой сессии. На Colab/Kaggle эта ячейка построит success rate по difficulty=1..5 для обеих политик.
